# DO Masked Card Classifier Training

Train a dedicated card-label classifier that uses scene masks from the segmenter as input guidance.

This notebook uses a 4-stage curriculum designed to improve precision without making training unnecessarily slow:
1. Stage 1: short warm-up on clean reference card crops.
2. Stage 2: train on generated augmented-card crops to learn color/value variations cheaply.
3. Stage 3: fine-tune on augmented-scene card crops using manual scene masks.
4. Stage 4: refine on augmented-scene card crops using masks predicted by the scene segmenter.

## Pipeline conventions and storage

- Only challenge data from this workspace is used.
- No pretrained weights are loaded (`torchvision` models use `weights=None`).
- Model parameter count is asserted to be <= 12M.
- Segmenter masks are used as supervision/input for classification preprocessing, not as a second prediction target.
- Training uses balanced sampling and capped augmented-card epochs so useful classes are seen often without cycling through every generated crop each epoch.
- Data sources:
  - `project/training_data/object_labels/reference_cards/reference_do.csv`
  - `project/training_data/object_labels/augmented_cards/aug.csv`
  - `project/training_data/object_labels/augmented_scenes/labels.json`
  - `project/training_data/training_images/reference_cards/`
  - `project/training_data/training_images/augmented_cards/`
  - `project/training_data/training_images/augmented_scenes/`
  - `project/training_data/training_masks/augmented_scenes/`
- Model artifacts are written to `project/models/`.

In [ ]:
from __future__ import annotations

import csv
import json
import random
import time
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from torchvision import models as tv_models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "project" / "training_data").exists() and (PROJECT_ROOT / "data").exists()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate workspace root containing project/training_data and data/.")
    PROJECT_ROOT = PROJECT_ROOT.parent

PROJECT_DIR = PROJECT_ROOT / "project"
TRAINING_DATA = PROJECT_DIR / "training_data"
MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_CSV = TRAINING_DATA / "object_labels" / "reference_cards" / "reference_do.csv"
SCENE_LABELS_PATH = TRAINING_DATA / "object_labels" / "augmented_scenes" / "labels.json"

REF_CARDS_DIR = TRAINING_DATA / "training_images" / "reference_cards"
SCENE_IMAGES_DIR = TRAINING_DATA / "training_images" / "augmented_scenes"
SCENE_MASKS_DIR = TRAINING_DATA / "training_masks" / "augmented_scenes"

SCENE_SEGMENTER_PATH = MODELS_DIR / "scene_segmenter_unet_small.pth"

CARD_CLASSIFIER_MODEL_PATH = MODELS_DIR / "card_classifier_masked_resnet18_do.pth"
CARD_CLASSIFIER_CLASSES_PATH = MODELS_DIR / "card_classifier_masked_resnet18_classes_do.npy"
CARD_CLASSIFIER_CONFIG_PATH = MODELS_DIR / "card_classifier_masked_resnet18_config_do.json"

for required_path in (REFERENCE_CSV, REF_CARDS_DIR, SCENE_LABELS_PATH, SCENE_IMAGES_DIR, SCENE_MASKS_DIR):
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required input: {required_path}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Training data: {TRAINING_DATA}")
print(f"Model output dir: {MODELS_DIR}")
print(f"Device: {device}")

## Classification Pipeline Building Blocks

This section defines:
- data loading and crop/mask preparation,
- scene-segmenter inference helper used to produce predicted masks for stage 3,
- a classification-specific backbone (ResNet18, random init),
- dataset and training utilities.

Design choices:
- input card size is fixed with letterboxing,
- the classifier consumes RGB + binary mask as a 4-channel tensor,
- the objective is classification-only (cross-entropy),
- segmenter masks are used for preprocessing/input guidance, not as an auxiliary prediction head.

In [ ]:
IMG_SIZE = 160
SEGMENTER_IMG_SIZE = 256
BBOX_MARGIN = 0.08
MASK_THRESHOLD = 0.50
VAL_SPLIT = 0.20

STAGE_1_EPOCHS = 4
STAGE_2_EPOCHS = 8
STAGE_3_EPOCHS = 6

STAGE_1_LR = 1e-3
STAGE_2_LR = 8e-4
STAGE_3_LR = 5e-4

WEIGHT_DECAY = 1e-4

BATCH_SIZE = 32 if device.type == "cuda" else 8
NUM_WORKERS = 0
USE_AMP = device.type == "cuda"

NORM_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
NORM_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


@dataclass(frozen=True)
class CardSample:
    stage: str
    label: str
    image_path: Path
    bbox: tuple[int, int, int, int] | None = None
    scene_mask_path: Path | None = None


def clean_label(label: str) -> str:
    return str(label).strip()


def resolve_project_path(path_value: str | Path) -> Path:
    path = Path(path_value)
    return path if path.is_absolute() else PROJECT_ROOT / path


def reference_crop_path(image_id: str) -> Path:
    tag, separator, crop_index = image_id.rpartition("_crop_")
    if separator == "":
        raise ValueError(f"Unexpected reference image_id: {image_id}")
    return REF_CARDS_DIR / tag / "crops" / f"crop_{crop_index}.jpg"


def is_reasonable_scene_bbox(bbox: tuple[int, int, int, int]) -> bool:
    x0, y0, x1, y1 = map(int, bbox)
    bw = x1 - x0
    bh = y1 - y0
    if bw < 28 or bh < 28:
        return False
    aspect = max(bw / max(1, bh), bh / max(1, bw))
    return aspect <= 4.5


def crop_with_margin(arr: np.ndarray, bbox: tuple[int, int, int, int], margin_fraction: float = BBOX_MARGIN) -> np.ndarray:
    x0, y0, x1, y1 = map(int, bbox)
    h, w = arr.shape[:2]
    bw = max(1, x1 - x0)
    bh = max(1, y1 - y0)
    margin = int(round(margin_fraction * max(bw, bh)))
    x0 = max(0, x0 - margin)
    y0 = max(0, y0 - margin)
    x1 = min(w, x1 + margin)
    y1 = min(h, y1 + margin)
    return arr[y0:y1, x0:x1]


def read_crop_image(sample: CardSample) -> np.ndarray:
    img_bgr = cv2.imread(str(sample.image_path), cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot read image: {sample.image_path}")
    if sample.bbox is not None:
        img_bgr = crop_with_margin(img_bgr, sample.bbox)
    return img_bgr


def letterbox_image_and_mask(
    img_bgr: np.ndarray,
    mask_u8: np.ndarray,
    size: int = IMG_SIZE,
    fill_image: int = 128,
    fill_mask: int = 0,
) -> tuple[np.ndarray, np.ndarray]:
    h, w = img_bgr.shape[:2]
    scale = size / max(h, w)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))

    img_resized = cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    mask_resized = cv2.resize(mask_u8, (new_w, new_h), interpolation=cv2.INTER_NEAREST)

    img_canvas = np.full((size, size, 3), fill_image, dtype=np.uint8)
    mask_canvas = np.full((size, size), fill_mask, dtype=np.uint8)

    y0 = (size - new_h) // 2
    x0 = (size - new_w) // 2
    img_canvas[y0:y0 + new_h, x0:x0 + new_w] = img_resized
    mask_canvas[y0:y0 + new_h, x0:x0 + new_w] = mask_resized
    return img_canvas, mask_canvas


def compose_masked_card_image(img_bgr: np.ndarray, mask_u8: np.ndarray, bg_fill: int = 128) -> np.ndarray:
    out = np.full_like(img_bgr, bg_fill)
    fg = mask_u8 > 0
    out[fg] = img_bgr[fg]
    return out


def card_input_to_tensor(img_bgr: np.ndarray, mask_u8: np.ndarray) -> torch.Tensor:
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img_chw = np.transpose(img_rgb, (2, 0, 1))
    img_chw = (img_chw - NORM_MEAN[:, None, None]) / NORM_STD[:, None, None]
    mask_ch = (mask_u8.astype(np.float32) / 255.0)[None, :, :]
    stacked = np.concatenate([img_chw, mask_ch], axis=0)
    return torch.from_numpy(stacked.astype(np.float32))


def count_trainable_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class SceneDoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class SceneUNetSmall(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = SceneDoubleConv(3, 32)
        self.enc2 = SceneDoubleConv(32, 64)
        self.enc3 = SceneDoubleConv(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = SceneDoubleConv(128, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = SceneDoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = SceneDoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = SceneDoubleConv(64, 32)
        self.out = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.out(d1)


def letterbox_for_segmenter(img_bgr: np.ndarray, target_size: int = SEGMENTER_IMG_SIZE, fill: int = 255) -> tuple[np.ndarray, dict[str, int]]:
    h, w = img_bgr.shape[:2]
    scale = min(target_size / w, target_size / h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    resized = cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    pad_w = target_size - new_w
    pad_h = target_size - new_h
    left = pad_w // 2
    top = pad_h // 2

    canvas = np.full((target_size, target_size, 3), fill, dtype=np.uint8)
    canvas[top:top + new_h, left:left + new_w] = resized
    meta = {
        "orig_w": w,
        "orig_h": h,
        "new_w": new_w,
        "new_h": new_h,
        "left": left,
        "top": top,
    }
    return canvas, meta


def unletterbox_probability(prob_lb: np.ndarray, meta: dict[str, int]) -> np.ndarray:
    left, top = int(meta["left"]), int(meta["top"])
    new_w, new_h = int(meta["new_w"]), int(meta["new_h"])
    orig_w, orig_h = int(meta["orig_w"]), int(meta["orig_h"])
    core = prob_lb[top:top + new_h, left:left + new_w]
    return cv2.resize(core, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)


def segment_scene_probability(img_bgr: np.ndarray, model: nn.Module, device_obj: torch.device) -> np.ndarray:
    img_lb, meta = letterbox_for_segmenter(img_bgr)
    x_rgb = cv2.cvtColor(img_lb, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    x = torch.from_numpy(np.transpose(x_rgb, (2, 0, 1))).float()
    mean = torch.tensor([0.485, 0.456, 0.406], dtype=x.dtype, device=x.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], dtype=x.dtype, device=x.device).view(3, 1, 1)
    x = (x - mean) / std
    x = x.unsqueeze(0).to(device_obj)
    with torch.no_grad():
        prob_lb = torch.sigmoid(model(x))[0, 0].cpu().numpy()
    return unletterbox_probability(prob_lb, meta)


class CardResNet18Classifier(nn.Module):
    def __init__(self, n_classes: int, input_channels: int = 4, dropout: float = 0.20):
        super().__init__()
        # Compliance: architecture only, random initialization, no pretrained weights.
        backbone = tv_models.resnet18(weights=None)
        old_conv = backbone.conv1
        backbone.conv1 = nn.Conv2d(
            input_channels,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False,
        )
        nn.init.kaiming_normal_(backbone.conv1.weight, mode="fan_out", nonlinearity="relu")
        in_features = backbone.fc.in_features
        backbone.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, n_classes))
        self.backbone = backbone

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


def sample_to_crop_and_mask(
    sample: CardSample,
    predicted_scene_probs: dict[str, np.ndarray] | None = None,
    mask_threshold: float = MASK_THRESHOLD,
) -> tuple[np.ndarray, np.ndarray]:
    img_crop = read_crop_image(sample)

    if sample.stage == "reference":
        mask_crop = np.full(img_crop.shape[:2], 255, dtype=np.uint8)
    elif sample.stage == "scene_manual":
        if sample.scene_mask_path is None:
            raise ValueError("scene_manual sample requires scene_mask_path.")
        scene_mask = cv2.imread(str(sample.scene_mask_path), cv2.IMREAD_GRAYSCALE)
        if scene_mask is None:
            raise FileNotFoundError(f"Cannot read scene mask: {sample.scene_mask_path}")
        if sample.bbox is None:
            raise ValueError("scene_manual sample requires bbox.")
        mask_crop = crop_with_margin(scene_mask, sample.bbox)
        mask_crop = np.where(mask_crop > 127, 255, 0).astype(np.uint8)
    elif sample.stage == "scene_predicted":
        if predicted_scene_probs is None:
            raise ValueError("predicted_scene_probs is required for scene_predicted samples.")
        if sample.bbox is None:
            raise ValueError("scene_predicted sample requires bbox.")
        prob_map = predicted_scene_probs.get(str(sample.image_path.resolve()))
        if prob_map is None:
            raise KeyError(f"Missing predicted probability map for scene: {sample.image_path}")
        prob_crop = crop_with_margin(prob_map, sample.bbox)
        mask_crop = np.where(prob_crop > mask_threshold, 255, 0).astype(np.uint8)
    else:
        raise ValueError(f"Unknown sample stage: {sample.stage}")

    if mask_crop.shape[:2] != img_crop.shape[:2]:
        mask_crop = cv2.resize(mask_crop, (img_crop.shape[1], img_crop.shape[0]), interpolation=cv2.INTER_NEAREST)

    if int(np.count_nonzero(mask_crop)) < 20:
        mask_crop = np.full(img_crop.shape[:2], 255, dtype=np.uint8)

    return img_crop, mask_crop


class CardMaskedDataset(Dataset):
    def __init__(
        self,
        samples: list[CardSample],
        label_to_index: dict[str, int],
        image_size: int = IMG_SIZE,
        augment: bool = False,
        predicted_scene_probs: dict[str, np.ndarray] | None = None,
    ):
        self.samples = samples
        self.label_to_index = label_to_index
        self.image_size = image_size
        self.augment = augment
        self.predicted_scene_probs = predicted_scene_probs

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        sample = self.samples[idx]
        img_crop, mask_crop = sample_to_crop_and_mask(sample, predicted_scene_probs=self.predicted_scene_probs)
        img_lb, mask_lb = letterbox_image_and_mask(img_crop, mask_crop, size=self.image_size)

        if self.augment:
            if random.random() < 0.5:
                img_lb = cv2.flip(img_lb, 1)
                mask_lb = cv2.flip(mask_lb, 1)
            if random.random() < 0.2:
                img_lb = cv2.convertScaleAbs(img_lb, alpha=1.0 + random.uniform(-0.10, 0.10), beta=random.uniform(-12, 12))

        masked_img = compose_masked_card_image(img_lb, mask_lb)
        x = card_input_to_tensor(masked_img, mask_lb)
        target = torch.tensor(self.label_to_index[sample.label], dtype=torch.long)
        return x, target


def make_loader(
    samples: list[CardSample],
    label_to_index: dict[str, int],
    batch_size: int,
    shuffle: bool,
    augment: bool,
    predicted_scene_probs: dict[str, np.ndarray] | None = None,
) -> tuple[DataLoader, CardMaskedDataset]:
    dataset = CardMaskedDataset(
        samples=samples,
        label_to_index=label_to_index,
        image_size=IMG_SIZE,
        augment=augment,
        predicted_scene_probs=predicted_scene_probs,
    )
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=device.type == "cuda",
    )
    return loader, dataset


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    ce_loss: nn.Module,
    scaler: torch.amp.GradScaler | None,
    amp_enabled: bool,
) -> dict[str, float]:
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    seen = 0

    for x, targets in loader:
        x = x.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            logits = model(x)
            loss = ce_loss(logits, targets)

        if scaler is not None and amp_enabled:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        batch_size_now = x.size(0)
        pred = torch.argmax(logits, dim=1)
        batch_acc = (pred == targets).float().mean().item()

        running_loss += loss.item() * batch_size_now
        running_acc += batch_acc * batch_size_now
        seen += batch_size_now

    return {
        "loss": running_loss / max(seen, 1),
        "cls_acc": running_acc / max(seen, 1),
    }


@torch.no_grad()
def evaluate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    ce_loss: nn.Module,
    amp_enabled: bool,
    return_predictions: bool = False,
):
    model.eval()
    running_loss = 0.0
    running_acc = 0.0
    seen = 0

    true_idx: list[int] = []
    pred_idx: list[int] = []

    for x, targets in loader:
        x = x.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            logits = model(x)
            loss = ce_loss(logits, targets)

        pred = torch.argmax(logits, dim=1)
        batch_acc = (pred == targets).float().mean().item()

        batch_size_now = x.size(0)
        running_loss += loss.item() * batch_size_now
        running_acc += batch_acc * batch_size_now
        seen += batch_size_now

        if return_predictions:
            true_idx.extend(targets.cpu().tolist())
            pred_idx.extend(pred.cpu().tolist())

    metrics = {
        "loss": running_loss / max(seen, 1),
        "cls_acc": running_acc / max(seen, 1),
    }

    if return_predictions:
        return metrics, np.array(true_idx, dtype=np.int64), np.array(pred_idx, dtype=np.int64)
    return metrics


@torch.no_grad()
def build_scene_probability_cache(scene_paths: list[Path], segmenter_path: Path) -> dict[str, np.ndarray]:
    if not segmenter_path.is_file():
        raise FileNotFoundError(f"Missing scene segmenter checkpoint: {segmenter_path}")

    segmenter = SceneUNetSmall().to(device)
    n_params = count_trainable_params(segmenter)
    assert n_params <= 12_000_000, f"Segmenter has {n_params:,} trainable params, exceeds 12M cap."
    print(f"[compliance] SceneUNetSmall: {n_params:,} trainable params")

    state_dict = torch.load(segmenter_path, map_location=device)
    segmenter.load_state_dict(state_dict)
    segmenter.eval()

    cache: dict[str, np.ndarray] = {}
    print(f"Building predicted scene masks for {len(scene_paths)} scenes...")
    for idx, scene_path in enumerate(scene_paths, start=1):
        img_bgr = cv2.imread(str(scene_path), cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(f"Cannot read scene image for prediction: {scene_path}")
        prob = segment_scene_probability(img_bgr, segmenter, device)
        cache[str(scene_path.resolve())] = prob.astype(np.float32)
        if idx % 100 == 0 or idx == len(scene_paths):
            print(f"  {idx}/{len(scene_paths)}")

    return cache


print("Classification helpers ready.")

In [ ]:
# Senior-dev improvement controls for this classifier pass.
# Keep the same checkpoint interface, but improve data curriculum, sampling, and augmentation.
from torch.utils.data import WeightedRandomSampler

AUG_CSV = TRAINING_DATA / "object_labels" / "augmented_cards" / "aug.csv"
AUG_CARDS_DIR = TRAINING_DATA / "training_images" / "augmented_cards"

for required_path in (AUG_CSV, AUG_CARDS_DIR):
    if not required_path.exists():
        raise FileNotFoundError(f"Missing augmented-card input: {required_path}")

if device.type == "cpu" and getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")

STAGE_1_EPOCHS = 0
STAGE_2_EPOCHS = 4
STAGE_3_EPOCHS = 5
STAGE_4_EPOCHS = 3

STAGE_1_LR = 1e-3
STAGE_2_LR = 1e-3
STAGE_3_LR = 7e-4
STAGE_4_LR = 4e-4

BATCH_SIZE = 32 if device.type == "cuda" else 16 if device.type == "mps" else 8
USE_AMP = device.type == "cuda"
BALANCED_SAMPLING = True
AUGMENTED_CARD_SAMPLES_PER_EPOCH = 4096 if device.type in {"cuda", "mps"} else 1536
EARLY_STOP_PATIENCE = 3
MIN_EPOCHS_PER_STAGE = 2


def _warp_affine_pair(img_bgr: np.ndarray, mask_u8: np.ndarray, angle: float, scale: float, tx: float, ty: float) -> tuple[np.ndarray, np.ndarray]:
    h, w = img_bgr.shape[:2]
    matrix = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    matrix[0, 2] += tx
    matrix[1, 2] += ty
    img_out = cv2.warpAffine(
        img_bgr,
        matrix,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(128, 128, 128),
    )
    mask_out = cv2.warpAffine(
        mask_u8,
        matrix,
        (w, h),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    return img_out, mask_out


def _warp_perspective_pair(img_bgr: np.ndarray, mask_u8: np.ndarray, max_jitter_fraction: float = 0.045) -> tuple[np.ndarray, np.ndarray]:
    h, w = img_bgr.shape[:2]
    jitter = max_jitter_fraction * min(h, w)
    src = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])
    dst = src + np.random.uniform(-jitter, jitter, size=(4, 2)).astype(np.float32)
    matrix = cv2.getPerspectiveTransform(src, dst)
    img_out = cv2.warpPerspective(
        img_bgr,
        matrix,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(128, 128, 128),
    )
    mask_out = cv2.warpPerspective(
        mask_u8,
        matrix,
        (w, h),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    return img_out, mask_out


def augment_card_image_and_mask(img_bgr: np.ndarray, mask_u8: np.ndarray, stage: str) -> tuple[np.ndarray, np.ndarray]:
    original_img = img_bgr.copy()
    original_mask = mask_u8.copy()
    h, w = img_bgr.shape[:2]

    if random.random() < 0.85:
        base_angle = random.uniform(-12.0, 12.0)
        if stage in {"augmented_card", "scene_manual", "scene_predicted"} and random.random() < 0.30:
            base_angle += random.choice([90.0, 180.0, 270.0])
        scale = random.uniform(0.92, 1.08)
        tx = random.uniform(-0.04, 0.04) * w
        ty = random.uniform(-0.04, 0.04) * h
        img_bgr, mask_u8 = _warp_affine_pair(img_bgr, mask_u8, base_angle, scale, tx, ty)

    if random.random() < 0.25:
        img_bgr, mask_u8 = _warp_perspective_pair(img_bgr, mask_u8)

    if random.random() < 0.65:
        alpha = random.uniform(0.82, 1.18)
        beta = random.uniform(-18.0, 18.0)
        img_bgr = cv2.convertScaleAbs(img_bgr, alpha=alpha, beta=beta)

    if random.random() < 0.35:
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[..., 1] *= random.uniform(0.80, 1.25)
        hsv[..., 2] *= random.uniform(0.85, 1.15)
        hsv = np.clip(hsv, 0, 255).astype(np.uint8)
        img_bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    if random.random() < 0.15:
        img_bgr = cv2.GaussianBlur(img_bgr, (3, 3), 0)

    if random.random() < 0.20:
        noise = np.random.normal(0.0, 4.0, size=img_bgr.shape).astype(np.float32)
        img_bgr = np.clip(img_bgr.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    if stage in {"scene_manual", "scene_predicted"} and random.random() < 0.30:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        if random.random() < 0.5:
            mask_u8 = cv2.erode(mask_u8, kernel, iterations=1)
        else:
            mask_u8 = cv2.dilate(mask_u8, kernel, iterations=1)

    if int(np.count_nonzero(mask_u8)) < 20:
        return original_img, original_mask
    return img_bgr, mask_u8


def sample_to_crop_and_mask(
    sample: CardSample,
    predicted_scene_probs: dict[str, np.ndarray] | None = None,
    mask_threshold: float = MASK_THRESHOLD,
) -> tuple[np.ndarray, np.ndarray]:
    img_crop = read_crop_image(sample)

    if sample.stage in {"reference", "augmented_card"}:
        mask_crop = np.full(img_crop.shape[:2], 255, dtype=np.uint8)
    elif sample.stage == "scene_manual":
        if sample.scene_mask_path is None:
            raise ValueError("scene_manual sample requires scene_mask_path.")
        scene_mask = cv2.imread(str(sample.scene_mask_path), cv2.IMREAD_GRAYSCALE)
        if scene_mask is None:
            raise FileNotFoundError(f"Cannot read scene mask: {sample.scene_mask_path}")
        if sample.bbox is None:
            raise ValueError("scene_manual sample requires bbox.")
        mask_crop = crop_with_margin(scene_mask, sample.bbox)
        mask_crop = np.where(mask_crop > 127, 255, 0).astype(np.uint8)
    elif sample.stage == "scene_predicted":
        if predicted_scene_probs is None:
            raise ValueError("predicted_scene_probs is required for scene_predicted samples.")
        if sample.bbox is None:
            raise ValueError("scene_predicted sample requires bbox.")
        prob_map = predicted_scene_probs.get(str(sample.image_path.resolve()))
        if prob_map is None:
            raise KeyError(f"Missing predicted probability map for scene: {sample.image_path}")
        prob_crop = crop_with_margin(prob_map, sample.bbox)
        mask_crop = np.where(prob_crop > mask_threshold, 255, 0).astype(np.uint8)
    else:
        raise ValueError(f"Unknown sample stage: {sample.stage}")

    if mask_crop.shape[:2] != img_crop.shape[:2]:
        mask_crop = cv2.resize(mask_crop, (img_crop.shape[1], img_crop.shape[0]), interpolation=cv2.INTER_NEAREST)

    if int(np.count_nonzero(mask_crop)) < 20:
        mask_crop = np.full(img_crop.shape[:2], 255, dtype=np.uint8)

    return img_crop, mask_crop


class CardMaskedDataset(Dataset):
    def __init__(
        self,
        samples: list[CardSample],
        label_to_index: dict[str, int],
        image_size: int = IMG_SIZE,
        augment: bool = False,
        predicted_scene_probs: dict[str, np.ndarray] | None = None,
    ):
        self.samples = samples
        self.label_to_index = label_to_index
        self.image_size = image_size
        self.augment = augment
        self.predicted_scene_probs = predicted_scene_probs

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        sample = self.samples[idx]
        img_crop, mask_crop = sample_to_crop_and_mask(sample, predicted_scene_probs=self.predicted_scene_probs)
        img_lb, mask_lb = letterbox_image_and_mask(img_crop, mask_crop, size=self.image_size)

        if self.augment:
            img_lb, mask_lb = augment_card_image_and_mask(img_lb, mask_lb, sample.stage)

        masked_img = compose_masked_card_image(img_lb, mask_lb)
        x = card_input_to_tensor(masked_img, mask_lb)
        target = torch.tensor(self.label_to_index[sample.label], dtype=torch.long)
        return x, target


def make_balanced_sampler(samples: list[CardSample], samples_per_epoch: int | None = None) -> WeightedRandomSampler:
    label_counts = Counter(sample.label for sample in samples)
    weights = torch.DoubleTensor([1.0 / label_counts[sample.label] for sample in samples])
    n_samples = len(samples) if samples_per_epoch is None else int(min(samples_per_epoch, len(samples)))
    generator = torch.Generator()
    generator.manual_seed(SEED)
    return WeightedRandomSampler(weights, num_samples=n_samples, replacement=True, generator=generator)


def make_loader(
    samples: list[CardSample],
    label_to_index: dict[str, int],
    batch_size: int,
    shuffle: bool,
    augment: bool,
    predicted_scene_probs: dict[str, np.ndarray] | None = None,
    balanced: bool = False,
    samples_per_epoch: int | None = None,
) -> tuple[DataLoader, CardMaskedDataset]:
    dataset = CardMaskedDataset(
        samples=samples,
        label_to_index=label_to_index,
        image_size=IMG_SIZE,
        augment=augment,
        predicted_scene_probs=predicted_scene_probs,
    )
    sampler = make_balanced_sampler(samples, samples_per_epoch=samples_per_epoch) if balanced and len(samples) > 0 else None
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle if sampler is None else False,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=device.type == "cuda",
    )
    return loader, dataset


def _autocast_device_type() -> str:
    return "cuda" if device.type == "cuda" else "cpu"


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    ce_loss: nn.Module,
    scaler: torch.amp.GradScaler | None,
    amp_enabled: bool,
) -> dict[str, float]:
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    seen = 0

    for x, targets in loader:
        x = x.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=_autocast_device_type(), dtype=torch.float16, enabled=amp_enabled):
            logits = model(x)
            loss = ce_loss(logits, targets)

        if scaler is not None and amp_enabled:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        batch_size_now = x.size(0)
        pred = torch.argmax(logits, dim=1)
        batch_acc = (pred == targets).float().mean().item()

        running_loss += loss.item() * batch_size_now
        running_acc += batch_acc * batch_size_now
        seen += batch_size_now

    return {
        "loss": running_loss / max(seen, 1),
        "cls_acc": running_acc / max(seen, 1),
    }


@torch.no_grad()
def evaluate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    ce_loss: nn.Module,
    amp_enabled: bool,
    return_predictions: bool = False,
):
    model.eval()
    running_loss = 0.0
    running_acc = 0.0
    seen = 0

    true_idx: list[int] = []
    pred_idx: list[int] = []

    for x, targets in loader:
        x = x.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.autocast(device_type=_autocast_device_type(), dtype=torch.float16, enabled=amp_enabled):
            logits = model(x)
            loss = ce_loss(logits, targets)

        pred = torch.argmax(logits, dim=1)
        batch_acc = (pred == targets).float().mean().item()

        batch_size_now = x.size(0)
        running_loss += loss.item() * batch_size_now
        running_acc += batch_acc * batch_size_now
        seen += batch_size_now

        if return_predictions:
            true_idx.extend(targets.cpu().tolist())
            pred_idx.extend(pred.cpu().tolist())

    metrics = {
        "loss": running_loss / max(seen, 1),
        "cls_acc": running_acc / max(seen, 1),
    }

    if return_predictions:
        return metrics, np.array(true_idx, dtype=np.int64), np.array(pred_idx, dtype=np.int64)
    return metrics


print("Improved classifier training controls ready.")
print(f"Augmented-card labels: {AUG_CSV}")
print(f"Augmented-card images: {AUG_CARDS_DIR}")
print(f"Device after accelerator check: {device}")
print(f"Batch size: {BATCH_SIZE}; AMP: {USE_AMP}")

## Build Stage Datasets

We build four sample groups:
- `reference`: clean card crops from reference labels.
- `augmented_card`: generated single-card augmentations from `augmented_cards/aug.csv`.
- `scene_manual`: scene card crops + manual scene masks.
- `scene_predicted`: scene card crops + masks from the trained scene segmenter.

Validation is held out from `scene_manual` only and reused for all training stages, because those crops are closest to the final scene-classification problem while still having trusted labels and masks.

In [ ]:
reference_samples: list[CardSample] = []
augmented_card_samples: list[CardSample] = []
scene_manual_samples: list[CardSample] = []

missing_files: list[str] = []
skipped_aug_labels = 0
skipped_aug_files = 0
skipped_scene_labels = 0
skipped_scene_boxes = 0
skipped_scene_masks = 0

# Stage 1 samples: reference card crops.
reference_labels: dict[str, str] = {}
with REFERENCE_CSV.open(newline="", encoding="utf-8") as csv_file:
    for row in csv.DictReader(csv_file):
        image_id = str(row["image_id"]).strip()
        label = clean_label(row["card"])
        if image_id and label:
            reference_labels[image_id] = label

for image_id, label in reference_labels.items():
    image_path = reference_crop_path(image_id)
    if image_path.is_file():
        reference_samples.append(CardSample("reference", label, image_path))
    else:
        missing_files.append(str(image_path))

# Stage 2 samples: generated augmented card crops.
valid_card_ext = (".jpg", ".jpeg", ".png")
with AUG_CSV.open(newline="", encoding="utf-8") as csv_file:
    for row in csv.DictReader(csv_file):
        image_id = str(row["image_id"]).strip()
        label = clean_label(row["card"])

        if not image_id or not label or label == "token":
            skipped_aug_labels += 1
            continue

        image_path = None
        for suffix in valid_card_ext:
            candidate_path = AUG_CARDS_DIR / f"{image_id}{suffix}"
            if candidate_path.is_file():
                image_path = candidate_path
                break

        if image_path is None:
            skipped_aug_files += 1
            missing_files.append(str(AUG_CARDS_DIR / f"{image_id}.jpg"))
            continue

        augmented_card_samples.append(CardSample("augmented_card", label, image_path))

# Stage 3 samples: scene crops with manual masks.
valid_mask_ext = {".png", ".jpg", ".jpeg"}
scene_mask_by_stem = {
    mask_path.stem: mask_path
    for mask_path in sorted(SCENE_MASKS_DIR.iterdir())
    if mask_path.suffix.lower() in valid_mask_ext
}

with SCENE_LABELS_PATH.open("r", encoding="utf-8") as labels_file:
    scene_metadata = json.load(labels_file)

for scene_entry in scene_metadata:
    scene_name = str(scene_entry.get("scene", "")).strip()
    default_scene_path = SCENE_IMAGES_DIR / f"{scene_name}.jpg"
    scene_path = resolve_project_path(scene_entry.get("image_path", default_scene_path))

    if not scene_path.is_file():
        missing_files.append(str(scene_path))
        continue

    scene_mask_path = scene_mask_by_stem.get(scene_path.stem) or scene_mask_by_stem.get(scene_name)

    for card in scene_entry.get("cards", []):
        label = clean_label(card.get("label", ""))
        bbox_raw = card.get("bbox", [])

        if not label or label == "token":
            skipped_scene_labels += 1
            continue

        if len(bbox_raw) != 4:
            skipped_scene_boxes += 1
            continue

        bbox = tuple(map(int, bbox_raw))
        if not is_reasonable_scene_bbox(bbox):
            skipped_scene_boxes += 1
            continue

        if scene_mask_path is None or not scene_mask_path.is_file():
            skipped_scene_masks += 1
            continue

        scene_manual_samples.append(
            CardSample(
                stage="scene_manual",
                label=label,
                image_path=scene_path,
                bbox=bbox,
                scene_mask_path=scene_mask_path,
            )
        )

if len(reference_samples) == 0:
    raise RuntimeError("No reference samples found. Run create_reference_cards_do.ipynb first.")
if len(augmented_card_samples) == 0:
    raise RuntimeError("No augmented-card samples found. Run create_augmented_data_do.ipynb first.")
if len(scene_manual_samples) == 0:
    raise RuntimeError("No scene-manual samples found. Run augmented-scene generation and mask creation first.")

all_labels = sorted({sample.label for sample in reference_samples + augmented_card_samples + scene_manual_samples})
label_encoder = LabelEncoder()
label_encoder.fit(all_labels)
label_to_index = {label: idx for idx, label in enumerate(label_encoder.classes_)}

scene_labels = np.array([sample.label for sample in scene_manual_samples])
scene_indices = np.arange(len(scene_manual_samples))
scene_label_counts = Counter(scene_labels)
min_scene_count = min(scene_label_counts.values())
stratify_scene = scene_labels if min_scene_count >= 2 else None
if stratify_scene is None:
    print("[warning] Some scene classes have only one sample; scene split is not stratified.")

scene_train_idx, scene_val_idx = train_test_split(
    scene_indices,
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=stratify_scene,
)

scene_train_samples = [scene_manual_samples[int(i)] for i in scene_train_idx]
scene_val_samples = [scene_manual_samples[int(i)] for i in scene_val_idx]

scene_predicted_train_samples = [
    CardSample(
        stage="scene_predicted",
        label=sample.label,
        image_path=sample.image_path,
        bbox=sample.bbox,
        scene_mask_path=None,
    )
    for sample in scene_train_samples
]

scene_paths_for_pred = sorted({sample.image_path.resolve() for sample in scene_predicted_train_samples})
predicted_scene_probs = build_scene_probability_cache(scene_paths_for_pred, SCENE_SEGMENTER_PATH)

augmented_samples_per_epoch = min(len(augmented_card_samples), AUGMENTED_CARD_SAMPLES_PER_EPOCH)

print(f"Reference samples:           {len(reference_samples)}")
print(f"Augmented-card samples:      {len(augmented_card_samples)}")
print(f"Aug-card samples/epoch:      {augmented_samples_per_epoch}")
print(f"Scene-manual samples:        {len(scene_manual_samples)}")
print(f"Scene-manual train samples:  {len(scene_train_samples)}")
print(f"Scene-manual val samples:    {len(scene_val_samples)}")
print(f"Scene-pred train samples:    {len(scene_predicted_train_samples)}")
print(f"Num classes:                 {len(label_encoder.classes_)}")
print(f"First classes:               {list(label_encoder.classes_[:10])}")
print(f"Skipped aug labels:          {skipped_aug_labels}")
print(f"Skipped aug files:           {skipped_aug_files}")
print(f"Skipped scene labels:        {skipped_scene_labels}")
print(f"Skipped scene boxes:         {skipped_scene_boxes}")
print(f"Skipped scene masks:         {skipped_scene_masks}")
if missing_files:
    print(f"Missing files skipped:       {len(missing_files)}")
    print("First missing files:")
    for item in missing_files[:5]:
        print(f"  {item}")

In [ ]:
# Quick visual check: image crop + supervision mask for each stage.
rng = np.random.default_rng(SEED)
preview_pool: list[CardSample] = []

preview_pool.extend(rng.choice(reference_samples, size=min(3, len(reference_samples)), replace=False).tolist())
preview_pool.extend(rng.choice(augmented_card_samples, size=min(3, len(augmented_card_samples)), replace=False).tolist())
preview_pool.extend(rng.choice(scene_train_samples, size=min(3, len(scene_train_samples)), replace=False).tolist())
preview_pool.extend(rng.choice(scene_predicted_train_samples, size=min(3, len(scene_predicted_train_samples)), replace=False).tolist())

cols = 4
rows = int(np.ceil(len(preview_pool) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 4.0 * rows))
axes = np.array(axes).reshape(-1)

for ax, sample in zip(axes, preview_pool):
    crop_bgr, mask_u8 = sample_to_crop_and_mask(sample, predicted_scene_probs=predicted_scene_probs)
    crop_lb, mask_lb = letterbox_image_and_mask(crop_bgr, mask_u8)
    overlay = cv2.cvtColor(crop_lb, cv2.COLOR_BGR2RGB).copy()
    overlay[mask_lb > 0] = (0.65 * overlay[mask_lb > 0] + 0.35 * np.array([0, 255, 255])).astype(np.uint8)

    ax.imshow(overlay)
    ax.set_title(f"{sample.stage}\n{sample.label}", fontsize=9)
    ax.axis("off")

for ax in axes[len(preview_pool):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Stage Training (Reference -> Augmented Cards -> Manual Masks -> Predicted Masks)

The classifier is trained sequentially with the same weights carried across stages:
1. bootstrap on clean references,
2. learn broad card appearance variation from generated augmented card crops,
3. adapt to scene crops with manual masks,
4. refine against predicted-mask noise from the segmenter.

Validation is always computed on held-out manual-scene samples.

Training-speed changes:
- class-balanced sampling keeps rare labels visible during each stage,
- the augmented-card stage samples a capped subset per epoch,
- later stages can stop early when validation accuracy plateaus.

Loss: classification-only cross-entropy.

In [ ]:
train_loader_ref, _ = make_loader(
    samples=reference_samples,
    label_to_index=label_to_index,
    batch_size=BATCH_SIZE,
    shuffle=True,
    augment=True,
    balanced=BALANCED_SAMPLING,
)

train_loader_augmented, _ = make_loader(
    samples=augmented_card_samples,
    label_to_index=label_to_index,
    batch_size=BATCH_SIZE,
    shuffle=True,
    augment=True,
    balanced=BALANCED_SAMPLING,
    samples_per_epoch=augmented_samples_per_epoch,
)

train_loader_manual, _ = make_loader(
    samples=scene_train_samples,
    label_to_index=label_to_index,
    batch_size=BATCH_SIZE,
    shuffle=True,
    augment=True,
    balanced=BALANCED_SAMPLING,
)

train_loader_predicted, _ = make_loader(
    samples=scene_predicted_train_samples,
    label_to_index=label_to_index,
    batch_size=BATCH_SIZE,
    shuffle=True,
    augment=True,
    predicted_scene_probs=predicted_scene_probs,
    balanced=BALANCED_SAMPLING,
)

val_loader, _ = make_loader(
    samples=scene_val_samples,
    label_to_index=label_to_index,
    batch_size=BATCH_SIZE,
    shuffle=False,
    augment=False,
)

model = CardResNet18Classifier(n_classes=len(label_encoder.classes_), input_channels=4).to(device)
model_params = count_trainable_params(model)
assert model_params <= 12_000_000, f"CardResNet18Classifier has {model_params:,} params, exceeds 12M cap"
print(f"[compliance] CardResNet18Classifier: {model_params:,} trainable params")

ce_loss = nn.CrossEntropyLoss(label_smoothing=0.05)

if device.type == "cuda":
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
else:
    scaler = None

stage_plan = [
    {
        "name": "stage1_reference",
        "epochs": STAGE_1_EPOCHS,
        "loader": train_loader_ref,
        "lr": STAGE_1_LR,
    },
    {
        "name": "stage2_augmented_cards",
        "epochs": STAGE_2_EPOCHS,
        "loader": train_loader_augmented,
        "lr": STAGE_2_LR,
    },
    {
        "name": "stage3_scene_manual_masks",
        "epochs": STAGE_3_EPOCHS,
        "loader": train_loader_manual,
        "lr": STAGE_3_LR,
    },
    {
        "name": "stage4_scene_predicted_masks",
        "epochs": STAGE_4_EPOCHS,
        "loader": train_loader_predicted,
        "lr": STAGE_4_LR,
    },
]

history: list[dict[str, float | int | str]] = []
best_val_acc = -1.0
best_state_dict = None
best_stage = ""
best_epoch = -1

global_start = time.perf_counter()

for stage_cfg in stage_plan:
    stage_name = str(stage_cfg["name"])
    stage_epochs = int(stage_cfg["epochs"])
    stage_loader = stage_cfg["loader"]
    stage_lr = float(stage_cfg["lr"])
    samples_per_epoch = len(stage_loader.sampler) if stage_loader.sampler is not None else len(stage_loader.dataset)

    if stage_epochs <= 0 or len(stage_loader.dataset) == 0:
        print(f"[skip] {stage_name}: no epochs or no samples")
        continue

    optimizer = torch.optim.AdamW(model.parameters(), lr=stage_lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
    )

    stage_best_val_acc = -1.0
    epochs_without_stage_improvement = 0

    print(
        f"\n[{stage_name}] epochs={stage_epochs}, lr={stage_lr:.2e}, "
        f"dataset_samples={len(stage_loader.dataset)}, samples_per_epoch={samples_per_epoch}"
    )

    for epoch in range(1, stage_epochs + 1):
        epoch_start = time.perf_counter()

        train_metrics = train_one_epoch(
            model=model,
            loader=stage_loader,
            optimizer=optimizer,
            ce_loss=ce_loss,
            scaler=scaler,
            amp_enabled=USE_AMP,
        )
        val_metrics = evaluate_one_epoch(
            model=model,
            loader=val_loader,
            ce_loss=ce_loss,
            amp_enabled=USE_AMP,
        )

        scheduler.step(val_metrics["cls_acc"])
        lr_now = float(optimizer.param_groups[0]["lr"])
        epoch_seconds = float(time.perf_counter() - epoch_start)

        row = {
            "stage": stage_name,
            "epoch": epoch,
            "train_loss": float(train_metrics["loss"]),
            "train_cls_acc": float(train_metrics["cls_acc"]),
            "val_loss": float(val_metrics["loss"]),
            "val_cls_acc": float(val_metrics["cls_acc"]),
            "lr": lr_now,
            "seconds": epoch_seconds,
            "samples_per_epoch": int(samples_per_epoch),
        }
        history.append(row)

        is_best = val_metrics["cls_acc"] > best_val_acc
        if is_best:
            best_val_acc = float(val_metrics["cls_acc"])
            best_state_dict = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            best_stage = stage_name
            best_epoch = epoch

        stage_improved = val_metrics["cls_acc"] > stage_best_val_acc + 1e-6
        if stage_improved:
            stage_best_val_acc = float(val_metrics["cls_acc"])
            epochs_without_stage_improvement = 0
        else:
            epochs_without_stage_improvement += 1

        print(
            f"  epoch {epoch:02d}/{stage_epochs} | "
            f"train_acc={train_metrics['cls_acc'] * 100:5.1f}% train_loss={train_metrics['loss']:.3f} | "
            f"val_acc={val_metrics['cls_acc'] * 100:5.1f}% val_loss={val_metrics['loss']:.3f} | "
            f"lr={lr_now:.2e} | {epoch_seconds:.1f}s"
            + (" | best" if is_best else "")
        )

        if (
            stage_name not in {"stage1_reference", "stage2_augmented_cards"}
            and epoch >= MIN_EPOCHS_PER_STAGE
            and epochs_without_stage_improvement >= EARLY_STOP_PATIENCE
        ):
            print(f"  [early-stop] {stage_name}: validation plateaued for {EARLY_STOP_PATIENCE} epochs")
            break

if best_state_dict is None:
    raise RuntimeError("Training did not produce any checkpoint.")

model.load_state_dict(best_state_dict)
training_seconds = time.perf_counter() - global_start

print("\nTraining complete.")
print(f"Best validation accuracy: {best_val_acc * 100:.2f}%")
print(f"Best checkpoint stage/epoch: {best_stage} / {best_epoch}")
print(f"Total training time: {training_seconds / 60:.2f} min")

In [ ]:
checkpoint = {
    "state_dict": model.state_dict(),
    "model_name": "CardResNet18Classifier",
    "n_classes": int(len(label_encoder.classes_)),
    "img_size": IMG_SIZE,
    "input_channels": 4,
}
torch.save(checkpoint, CARD_CLASSIFIER_MODEL_PATH)
np.save(CARD_CLASSIFIER_CLASSES_PATH, label_encoder.classes_)

config = {
    "model_file": CARD_CLASSIFIER_MODEL_PATH.name,
    "classes_file": CARD_CLASSIFIER_CLASSES_PATH.name,
    "image_size": IMG_SIZE,
    "normalization_mean": NORM_MEAN.tolist(),
    "normalization_std": NORM_STD.tolist(),
    "bbox_margin": BBOX_MARGIN,
    "mask_threshold": MASK_THRESHOLD,
    "input_mode": "rgb_plus_mask_channel",
    "masked_background_fill": 128,
    "architecture": "torchvision_resnet18",
    "pretrained": False,
    "optimizer": "AdamW",
    "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE,
    "device": str(device),
    "balanced_sampling": BALANCED_SAMPLING,
    "augmented_card_samples_per_epoch": int(augmented_samples_per_epoch),
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "min_epochs_per_stage": MIN_EPOCHS_PER_STAGE,
    "augmentation_policy": {
        "random_rotation_degrees": 12,
        "right_angle_rotation_probability": 0.30,
        "perspective_probability": 0.25,
        "brightness_contrast_probability": 0.65,
        "hsv_probability": 0.35,
        "blur_probability": 0.15,
        "noise_probability": 0.20,
        "mask_jitter_probability_scene_stages": 0.30,
        "horizontal_flip": False,
    },
    "stage_plan": [
        {
            "stage": "stage1_reference",
            "epochs": STAGE_1_EPOCHS,
            "learning_rate": STAGE_1_LR,
            "n_samples": len(reference_samples),
            "samples_per_epoch": len(train_loader_ref.sampler) if train_loader_ref.sampler is not None else len(train_loader_ref.dataset),
        },
        {
            "stage": "stage2_augmented_cards",
            "epochs": STAGE_2_EPOCHS,
            "learning_rate": STAGE_2_LR,
            "n_samples": len(augmented_card_samples),
            "samples_per_epoch": len(train_loader_augmented.sampler) if train_loader_augmented.sampler is not None else len(train_loader_augmented.dataset),
        },
        {
            "stage": "stage3_scene_manual_masks",
            "epochs": STAGE_3_EPOCHS,
            "learning_rate": STAGE_3_LR,
            "n_samples": len(scene_train_samples),
            "samples_per_epoch": len(train_loader_manual.sampler) if train_loader_manual.sampler is not None else len(train_loader_manual.dataset),
        },
        {
            "stage": "stage4_scene_predicted_masks",
            "epochs": STAGE_4_EPOCHS,
            "learning_rate": STAGE_4_LR,
            "n_samples": len(scene_predicted_train_samples),
            "samples_per_epoch": len(train_loader_predicted.sampler) if train_loader_predicted.sampler is not None else len(train_loader_predicted.dataset),
        },
    ],
    "n_classes": int(len(label_encoder.classes_)),
    "class_names": [str(c) for c in label_encoder.classes_],
    "validation_samples": int(len(scene_val_samples)),
    "best_val_accuracy": float(best_val_acc),
    "best_stage": best_stage,
    "best_epoch": int(best_epoch),
    "seed": SEED,
}

with CARD_CLASSIFIER_CONFIG_PATH.open("w", encoding="utf-8") as config_file:
    json.dump(config, config_file, indent=2)

print(f"Saved model checkpoint: {CARD_CLASSIFIER_MODEL_PATH}")
print(f"Saved class names:      {CARD_CLASSIFIER_CLASSES_PATH}")
print(f"Saved config:           {CARD_CLASSIFIER_CONFIG_PATH}")

## Validation Diagnostics

We evaluate the best checkpoint on the held-out manual-scene validation set.

Reported diagnostics:
- classification report and confusion matrix,
- stage-wise learning curves (accuracy and loss),
- qualitative errors with mask-guided classifier input overlays.

In [ ]:
val_metrics, val_true_idx, val_pred_idx = evaluate_one_epoch(
    model=model,
    loader=val_loader,
    ce_loss=ce_loss,
    amp_enabled=USE_AMP,
    return_predictions=True,
)

true_labels = label_encoder.inverse_transform(val_true_idx)
pred_labels = label_encoder.inverse_transform(val_pred_idx)

print(f"Validation accuracy: {val_metrics['cls_acc'] * 100:.2f}%")
print(classification_report(true_labels, pred_labels, zero_division=0))

fig_size = max(8, 0.30 * len(label_encoder.classes_))
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
ConfusionMatrixDisplay.from_predictions(
    true_labels,
    pred_labels,
    labels=label_encoder.classes_,
    xticks_rotation=90,
    colorbar=False,
    ax=ax,
 )
ax.set_title(f"Masked classifier validation accuracy: {val_metrics['cls_acc'] * 100:.1f}%")
plt.tight_layout()
plt.show()

if len(history) > 0:
    epochs_x = np.arange(1, len(history) + 1)
    train_acc_curve = [float(row["train_cls_acc"]) for row in history]
    val_acc_curve = [float(row["val_cls_acc"]) for row in history]
    train_loss_curve = [float(row["train_loss"]) for row in history]
    val_loss_curve = [float(row["val_loss"]) for row in history]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(epochs_x, train_acc_curve, marker="o", label="train acc")
    axes[0].plot(epochs_x, val_acc_curve, marker="o", label="val acc")
    axes[0].set_ylim(0.0, 1.0)
    axes[0].set_title("Classification accuracy")
    axes[0].set_xlabel("Global epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(epochs_x, train_loss_curve, marker="o", label="train loss")
    axes[1].plot(epochs_x, val_loss_curve, marker="o", label="val loss")
    axes[1].set_title("Cross-entropy loss")
    axes[1].set_xlabel("Global epoch")
    axes[1].set_ylabel("Loss")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    for idx, row in enumerate(history, start=1):
        stage_name = str(row["stage"]).replace("stage", "S")
        axes[0].annotate(stage_name, (idx, float(row["val_cls_acc"])), fontsize=7, alpha=0.65)

    plt.tight_layout()
    plt.show()

In [ ]:
wrong_indices = [
    idx
    for idx, (true_label, pred_label) in enumerate(zip(true_labels, pred_labels))
    if true_label != pred_label
]

print(f"Wrong validation predictions: {len(wrong_indices)} / {len(scene_val_samples)}")

if wrong_indices:
    show_count = min(12, len(wrong_indices))
    cols = 4
    rows = int(np.ceil(show_count / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 3.8 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, idx in zip(axes, wrong_indices[:show_count]):
        sample = scene_val_samples[int(idx)]
        crop_bgr, manual_mask = sample_to_crop_and_mask(sample, predicted_scene_probs=None)
        crop_lb, manual_mask_lb = letterbox_image_and_mask(crop_bgr, manual_mask)

        masked_img = compose_masked_card_image(crop_lb, manual_mask_lb)
        x = card_input_to_tensor(masked_img, manual_mask_lb).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[0]
            pred_index = int(torch.argmax(probs).item())
            pred_conf = float(probs[pred_index].item())

        vis = cv2.cvtColor(masked_img, cv2.COLOR_BGR2RGB)
        true_label = true_labels[int(idx)]
        pred_label = pred_labels[int(idx)]

        ax.imshow(vis)
        ax.set_title(
            f"true={true_label}\npred={pred_label} ({pred_conf:.2f})",
            fontsize=8,
            color="red",
        )
        ax.axis("off")

    for ax in axes[show_count:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()